# Download MASC Dataset with Live Status

This notebook downloads the MASC (Massive Arabic Speech Corpus) dataset used in this project.

Source confirmed against `.intermediate_data/MASC-Arabic2/README.md` (byte-identical dataset
card match, verified via the HF Hub API): the correct Hugging Face repo is
`MohamedRashad/MASC-Arabic` -- **not** `pain/MASC`, which the card's prose credits as the
"Original Dataset Repo" but is actually a different card with different README contents.

Recorded splits from that same local dataset card (`dataset_info` block), for reference:

- `train`: 875,873 examples
- `validation`: 19,521 examples
- `test`: 18,006 examples
- `download_size`: ~184.87 GB
- `dataset_size` (decompressed): ~209.19 GB

Downloads straight into `MASC-Arabic2/` at the repo root, which is symlinked to
`/workspace/asr/Palestinian-ASR/MASC-Arabic2` (the project's consistent large-volume storage),
so this does not touch the small root disk.

After this completes, run `scripts/filter_masc_c_only.py --source MASC-Arabic2 --output data/masc_c_only`
per `DATA_CURATION.md` to keep only `type == "c"` (clean) rows.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from time import sleep, time

from huggingface_hub import snapshot_download

repo_id = 'MohamedRashad/MASC-Arabic'
local_dir = Path('/root/asr/Palestinian-ASR/MASC-Arabic2')
local_dir.mkdir(parents=True, exist_ok=True)

def folder_size_bytes(path: Path) -> int:
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total

def download_dataset():
    return snapshot_download(
        repo_id=repo_id,
        repo_type='dataset',
        local_dir=str(local_dir),
        max_workers=4,
    )

print(f'Starting download: {repo_id}')
start = time()
with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(download_dataset)
    last_size = -1
    while not future.done():
        size = folder_size_bytes(local_dir)
        if size != last_size:
            elapsed = time() - start
            print(f'[{elapsed:7.1f}s] downloading... current size: {size / (1024 ** 3):.2f} GiB')
            last_size = size
        else:
            elapsed = time() - start
            print(f'[{elapsed:7.1f}s] still downloading...')
        sleep(15)

    path = future.result()
elapsed = time() - start
print(f'Download complete in {elapsed:.1f}s')
print('Downloaded to:', path)
print(f'Final size: {folder_size_bytes(local_dir) / (1024 ** 3):.2f} GiB')


In [1]:
from pathlib import Path

data_dir = Path('/root/asr/Palestinian-ASR/MASC-Arabic2/data')
shards = sorted(data_dir.glob('*.parquet'))
print(f'{len(shards)} parquet shards found under {data_dir}')
for s in shards[:10]:
    print(s.name)


417 parquet shards found under /root/asr/Palestinian-ASR/MASC-Arabic2/data
test-00000-of-00009.parquet
test-00001-of-00009.parquet
test-00002-of-00009.parquet
test-00003-of-00009.parquet
test-00004-of-00009.parquet
test-00005-of-00009.parquet
test-00006-of-00009.parquet
test-00007-of-00009.parquet
test-00008-of-00009.parquet
train-00000-of-00400.parquet
